In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 15.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [ ]:
import json
import re
import os
import torch
import torchaudio
import numpy as np
import soundfile as sf
from tqdm import tqdm
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets, Audio
from transformers import Wav2Vec2Tokenizer, Wav2Vec2ForCTC, Wav2Vec2Processor, TrainingArguments, Trainer

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
def clean_text(text):
    text = text.replace("\n", " ")
    # # Remove speaker names (e.g., "RACHEL:" or "ROSS & PHOEBE:")
    # text = re.sub(r"^[A-Z &]+ ?: ?", "", text, flags=re.MULTILINE)
    # Remove speaker names (e.g., "RACHEL:" or "ROSS & PHOEBE:")
    text = re.sub(r"[A-Z &]+: ?", " ", text)

    # Expand contractions
    contractions = {
        "it's": "it is", "you're": "you are", "i'm": "i am", "don't": "do not",
        "won't": "will not", "she's": "she is", "he's": "he is", "we're": "we are",
        "they're": "they are", "can't": "cannot", "didn't": "did not", "i've": "i have",
        "wasn't": "was not", "isn't": "is not", "aren't": "are not", "let's": "let us",
        "what's": "what is", "there's": "there is", "that's": "that is", "it'd": "it would",
        "you'd": "you would", "i'd": "i would"
    }
    for contraction, expanded in contractions.items():
        text = re.sub(rf"\b{re.escape(contraction)}\b", expanded, text, flags=re.IGNORECASE)

    # Remove all punctuation
    text = re.sub(r"[^\w\s]", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Convert to uppercase
    text = text.upper()

    return text


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## ~~We directly use 16K downsampled data, so we don't have to downsample them~~
## 16K encountered problem when preprocessing

In [ ]:
# This is a cleaned json file for audio clips in part_1
data_split = '/content/drive/MyDrive/EE6366_Speech_Speaker_Recognition/VIOLIN_Audio_Data/data_subtitles_TV.json'
audio_dir = "/content/drive/MyDrive/EE6366_Speech_Speaker_Recognition/VIOLIN_Audio_Data/audio_clips/"


# This is a cleaned json file for audio clips in part_1, sperate into segments defined by subtitles
# part_1_short_split = '/content/drive/MyDrive/EE6366_Speech_Speaker_Recognition/VIOLIN_Audio_Data/data_subtitles_split_1_short_TV.json'
# audio_dir = "/content/drive/MyDrive/EE6366_Speech_Speaker_Recognition/VIOLIN_Audio_Data/audio_clips_part_1_short/"

with open(data_split) as f:
  data = json.load(f)

In [ ]:
split_data = {
    "train": [],
    "validate": [],
    "test": []
}

for key in tqdm(data,desc="Preprocessing dataset"):
  try:
    data_path = audio_dir + key + ".flac"
    assert os.path.exists(data_path), f"File not found: {data_path}"

    # array = downsample_audio(data_path)
    split = data[key]['split']  # Get the split (train/validation/test)

    # Append data to the corresponding split
    split_data[split].append(
      {
        "file": data_path,
        "name": key,
        # "audio": {
        #   "array": array,
        #   "sampling_rate": 16000,
        # },
        "text": clean_text(data[key]['sub']),
        # "duration": data[key]['duration'],
        # "speech_duration": data[key]['speech_duration'],
      }
    )
  except:
    continue


# Create DatasetDict from the split_data
dataset = DatasetDict(
    {split: Dataset.from_list(data_list) for split, data_list in split_data.items()}
)
dataset.save_to_disk("/content/drive/MyDrive/EE6366_Speech_Speaker_Recognition/VIOLIN_Audio_Data/dataset_TV")

In [ ]:
dataset = []

In [ ]:
dataset= DatasetDict.load_from_disk("/content/drive/MyDrive/EE6366_Speech_Speaker_Recognition/VIOLIN_Audio_Data/dataset_TV")
# dataset= DatasetDict.load_from_disk("/content/drive/MyDrive/EE6366_Speech_Speaker_Recognition/VIOLIN_Audio_Data/dataset_TV_16K")